### 1. Setup & Library Imports

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns; sns.set_style('whitegrid')
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import plotly.express as px
plt.rcParams['figure.dpi'] = 150

### 2. Load Dataset & Exploratory Data Analysis 

#### 2.1 Loading original datasets

In [10]:
df_train = pd.read_csv('fraudTrain.csv')
df_test = pd.read_csv('fraudTest.csv')

#### 2.2 Adding datasets flags

In [11]:
df_train['data_type'] = 'train'
df_test['data_type'] = 'test'

#### 2.3 Creating the masters dataset

In [12]:
transactions_master = pd.concat([df_train, df_test], ignore_index=True)
print(f"✅ Master dataset: {transactions_master.shape}")
print(transactions_master.head(3))

✅ Master dataset: (1604294, 24)
   ID trans_date_trans_time        cc_num                         merchant  \
0   0         1/1/2019 0:00  2.703190e+15       fraud_Rippin, Kub and Mann   
1   1         1/1/2019 0:00  6.304230e+11  fraud_Heller, Gutmann and Zieme   
2   2         1/1/2019 0:00  3.885950e+13             fraud_Lind-Buckridge   

        category     amt      first     last gender  \
0       misc_net    4.97   Jennifer    Banks      F   
1    grocery_pos  107.23  Stephanie     Gill      F   
2  entertainment  220.11     Edward  Sanchez      M   

                         street  ...      long city_pop  \
0                561 Perry Cove  ...  -81.1781     3495   
1  43039 Riley Greens Suite 393  ... -118.2105      149   
2      594 White Dale Suite 530  ... -112.2620     4154   

                                 job        dob  \
0          Psychologist, counselling   3/9/1988   
1  Special educational needs teacher  6/21/1978   
2        Nature conservation officer  1/19/1

#### 2.4 Fraud Rate Check

In [13]:
print("\nFraud distribution:")
print(transactions_master.is_fraud.value_counts(normalize=True))
print(f"Train fraud: {df_train.is_fraud.mean():.3%}")
print(f"Test fraud:  {df_test.is_fraud.mean():.3%}")


Fraud distribution:
is_fraud
0    0.994919
1    0.005081
Name: proportion, dtype: float64
Train fraud: 0.573%
Test fraud:  0.386%


#### 2.5 Key Dimensions

In [14]:
print(f"\nCards: {transactions_master.cc_num.nunique():,}")
print(f"Merchants: {transactions_master.merchant.nunique():,}")
print(f"Train/Test split: {transactions_master.data_type.value_counts()}")


Cards: 959
Merchants: 693
Train/Test split: data_type
train    1048575
test      555719
Name: count, dtype: int64


#### 2.6 Summary Statistics

In [15]:
print("\nTransaction amounts:")
print(transactions_master.amt.describe())


Transaction amounts:
count    1.604294e+06
mean     6.997209e+01
std      1.588492e+02
min      1.000000e+00
25%      9.640000e+00
50%      4.739000e+01
75%      8.304000e+01
max      2.894890e+04
Name: amt, dtype: float64


#### 2.7 Data Quality

In [16]:
print("\nMissing values:")
print(transactions_master.isnull().sum())


Missing values:
ID                       0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
data_type                0
dtype: int64


In [7]:
df_train.head(5)

,ID,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud,data_type
0,0,1/1/2019 0:00,2.703190e+15,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,-81.1781,3495,"Psychologist, counselling",3/9/1988,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0,Train
1,1,1/1/2019 0:00,6.304230e+11,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,-118.2105,149,Special educational needs teacher,6/21/1978,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0,Train
2,2,1/1/2019 0:00,3.885950e+13,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,-112.2620,4154,Nature conservation officer,1/19/1962,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0,Train
3,3,1/1/2019 0:01,3.534090e+15,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,-112.1138,1939,Patent attorney,1/12/1967,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0,Train
4,4,1/1/2019 0:03,3.755340e+14,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,-79.4629,99,Dance movement psychotherapist,3/28/1986,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0,Train
